# Episode 01: MLX Inference — Running LLMs Locally on Apple Silicon

> *"The best model is the one you actually control."*

---

## Series Introduction

Welcome to **LLM Lab** — a hands-on engineering series for people who want to understand what's actually happening when they run large language models. This isn't a prompt engineering course. This is for engineers who want to know what's inside the box — the memory layout, the math, the hardware constraints, and the tools that let you control all of it.

**Who this is for:** Engineers with Python proficiency who care about what happens at the hardware level. You know what floating-point formats are, you've used virtual environments, and you have a conceptual understanding of neural networks at the "it does matrix math" level. You do *not* need prior experience with HuggingFace or Apple ML frameworks — those are introduced here.

**Prerequisites:**
- Python proficiency (reading and running scripts)
- Virtual environments (`source .venv/bin/activate`)
- Basic command-line usage
- Awareness of floating-point formats (fp32, bf16)
- Conceptual understanding of neural networks at the "it does matrix math" level

---

⚠️ **Run cells in order.** Code cells talk to a local MLX server on port 8800. Make sure your MLX server is running before executing code cells:

```bash
mlx_lm.server --model mlx-community/Qwen3-32B-4bit --port 8800
```

All you need: `pip install openai psutil mlx`

## Hardware Check

Let's verify your environment is ready. The cell below runs `setup_check.py` to confirm your Apple Silicon hardware, Python environment, and MLX installation are properly configured.

In [ ]:
!python setup_check.py

## 1. The Big Picture

Imagine you've spent years writing firmware for microcontrollers — you know exactly where every byte lives, you profile cycle counts, you care deeply about what happens at the hardware level. Now someone hands you a "magic black box" that generates human language, and everyone just says *"just vibe-code it, bro."* Eventually you learn there's an API, and behind that API there's a model, and behind that model there's actual math running on actual hardware.

That's not good enough for surface-level understanding. You want to know what's inside the box — and what you can do with it once you understand the tools surrounding it.

Here's the good news: **a transformer is not magic.** It's a small set of matrix multiplies and additions, repeated dozens of times. "122 billion parameters" means 122 billion numbers sitting in weight matrices. Inference reads those weights from RAM and multiplies them with your input. That's it.

Here's the thing: **you can open the box.** On your Apple Silicon Mac, you can run large language models — the same scale as what powers frontier AI products — entirely locally, with no cloud, no API keys, no data leaving your machine.

### Why does this matter for you?

- **Privacy:** your prompts never leave your machine
- **Latency:** no round-trip to a datacenter
- **Cost:** $0/token after hardware
- **Control:** you choose the model, the quantization, the context length, the sampling parameters
- **Understanding:** you can profile, instrument, and modify everything

Let's go from zero to running an LLM locally.

### Unified Memory: The Hardware Foundation

```
Traditional GPU Setup          Apple Silicon (M4 Max)
─────────────────────          ──────────────────────

CPU RAM (DDR5)                 Unified Memory (LPDDR5X)
    │                          ┌──────────────────────────┐
    │ PCIe bus                 │  CPU cores               │
    ▼                          │  GPU cores (40)          │
GPU VRAM (HBM)                 │  Neural Engine (38 TOPS) │
                               │  Media Engine            │
Model weights live here.       │                          │
Only GPU can access them.      │  All share same RAM      │
                               └──────────────────────────┘
                               Model weights live here.
                               Everything can access them.
```

**Key Insight:** On a discrete GPU, you're limited by VRAM (typically 24GB). On M4 Max with 128GB unified memory, your "VRAM" *is* your system RAM. You simply cannot run very large models on most single-GPU setups without expensive multi-GPU configurations.

### Memory Bandwidth: The Tradeoff

| Chip | Memory Bandwidth |
|------|------------------|
| **M4 Max** (40-core GPU) | **546 GB/s** |
| H100 SXM5 | 3.35 TB/s |
| H100 PCIe | ~2 TB/s |

For inference (which is memory-bandwidth-bound), you're at roughly 1/6th the throughput of server-class hardware. But you're running *locally*, *privately*, *free*.

**The embedded analogy:** Think of a traditional GPU server like a DSP coprocessor on a PCB — it has its own fast SRAM, but getting data *to* it crosses a slow bus. Apple Silicon is more like a tightly integrated SoC where your NPU, CPU, and GPU all read from the same address space. That's not a marketing claim — it's the actual memory topology.

### MLX: Apple's Array Framework

**MLX** is Apple's array framework built specifically for this unified memory architecture. It's not PyTorch ported to Metal. It was designed from scratch for unified memory, with **lazy evaluation** and **dynamic graph construction** baked in from day one.

**Neural Engine note:** MLX inference runs primarily on the GPU cores. The 16-core Neural Engine (ANE) is used by Core ML — Apple's higher-level inference API — for specific optimized operations, but it is not directly programmable through MLX's compute graph.

## 2. Core Concepts

### 2.1 Lazy Evaluation: How MLX Schedules Work

MLX uses **lazy evaluation** — a concept you might recognize from functional programming or certain HDL simulation semantics. When you write operations, the computation graph is built up silently, then flushed when you actually need the values.

This isn't just a quirk — it's what allows MLX to optimize across operations, fuse kernels, and schedule work efficiently across CPU/GPU without you thinking about it.

**The embedded systems analogy:** It's like DMA with scatter-gather lists. You describe *what* should happen (the transfers), then kick it off in one shot. The hardware figures out the optimal order. You don't block on each individual operation.

For inference, this means MLX can overlap compute and memory fetches, keep the GPU fed, and minimize idle time — all automatically.

In [ ]:
import mlx.core as mx

x = mx.array([1.0, 2.0, 3.0])
y = x * 2          # Nothing computed yet
z = y + 1          # Still nothing computed
print(z)           # NOW the computation runs

### 2.2 Quantization: Lossy Compression for Neural Weights

A language model is, at its core, a massive tensor of floating-point numbers. At full precision (fp32), each weight costs 4 bytes. At bfloat16, 2 bytes. At 4-bit integer quantization, 0.5 bytes.

**The key insight:** Neural network weights are surprisingly robust to precision reduction. The model learned to approximate a function — and that approximation is still valid if you represent the weights with less precision, *as long as you're careful about which weights matter most*.

```
Precision   Bytes/param   122B model  Quality
──────────────────────────────────────────────
fp32        4.0           488 GB      Reference
bf16        2.0           244 GB      ~identical
8-bit       1.0           122 GB      ~99% quality
6-bit       0.75           92 GB      ~98% quality
NVFP4       ~0.5           ~65 GB     ~95-98% quality (float format, better than affine INT4)
4-bit       0.5            61 GB      ~95-97% quality
3-bit       0.375          46 GB      noticeable degradation
2-bit       0.25           31 GB      significant degradation
```

**The hardware engineer's mental model:** Think of it like ADC resolution. 16-bit audio vs 8-bit audio — you lose dynamic range and introduce quantization noise, but for most content it's fine. Go to 4-bit and you start hearing artifacts. Go to 2-bit and it's clearly degraded. Same idea here, except your signal is a probability distribution over tokens.

### Group Quantization

MLX supports quantization natively and uses **group quantization** — rather than using a single scale factor for the entire weight matrix, it divides weights into small groups (e.g., 64 weights) and quantizes each group independently. This dramatically reduces quantization error.

Think of it like calibrating an ADC: you can share one gain/offset calibration across 64 channels, or calibrate every 16 channels independently. With shared calibration, channels with different characteristics get the wrong correction. Smaller groups = more independent calibrations = less systematic error across billions of parameters.

### NVFP4: Our Quantization Format

**NVFP4** uses a floating-point 4-bit format:
- **E2M1** weights (2-bit exponent, 1-bit mantissa)
- **E4M3** float scales
- **group_size = 16**

This places more representable values near zero — where neural network weights cluster — delivering better quality than affine INT4 at the same bit width. Module 1C covers advanced quantization formats (NVFP4, MXFP4) in detail.

### 2.3 How a Transformer Works: Layers, Heads, and the Forward Pass

A transformer is a stack of identical-ish blocks called **layers**, one on top of the other. Input goes in the bottom, flows through each block in sequence, output comes out the top.

**Layers** are like a multi-stage filter chain — each stage does a small transform on the data, and the cumulative effect across all the layers is powerful. Each layer has two main parts: an **attention mechanism** and a **feed-forward network (FFN)**.

An FFN is just a stack of matrix multiplies with a nonlinear activation in between — the simplest kind of neural network. No recurrence, no attention — data flows straight through in one direction. In a transformer, the FFN processes each token independently after attention has mixed information across tokens.

**Heads** are inside the attention mechanism. Instead of one monolithic attention computation, the model splits the work into parallel "heads" — each head looks at the same tokens but learns to focus on different patterns. One head might track subject-verb relationships, another tracks proximity, another tracks tone. The results get concatenated back together. It's like having 24 parallel pattern detectors instead of one.

**Head dimension** is the vector size each head works with. This number shows up in memory formulas — particularly the KV cache calculation.

**SwiGLU activation:** Modern transformers (Qwen, Llama, Mistral) use a gated FFN called SwiGLU instead of a plain ReLU FFN. The idea: split the computation into two parallel paths — a "gate" path and an "up" path — multiply them element-wise, then project down. This requires three weight matrices (W_gate, W_up, W_down) per layer instead of two. SwiGLU empirically improves model quality over ReLU at similar parameter counts.

### The Forward Pass

Here's what actually happens when you call `generate()`:

```
Input tokens (integer IDs)
         │
         ▼
Embedding lookup           ← read rows from embedding matrix (weights in RAM)
         │
         ▼
For each transformer layer:
  ├── Layer norm            ← normalize activations
  ├── Attention:
  │   ├── Q = x @ Wq        ← matrix multiply (GPU compute)
  │   ├── K = x @ Wk        ← matrix multiply
  │   ├── V = x @ Wv        ← matrix multiply
  │   ├── scores = Q @ K^T  ← attention scores
  │   ├── attn = softmax(scores) @ V
  │   └── cache K, V        ← store in KV cache (RAM write)
  ├── FFN (feed-forward):
  │   ├── gate = x @ W_gate ← for SwiGLU activation
  │   ├── up = x @ W_up
  │   └── down = (gate * up) @ W_down
  └── Residual connection
         │
         ▼
Final layer norm + LM head  ← project to vocab size (32K-150K tokens)
         │
         ▼
Softmax → sample → next token ID
```

Every weight matrix (Wq, Wk, Wv, W_gate, W_up, W_down) in every layer is a chunk of those billions of parameters — and must be read from RAM for every token.

### Throughput Calculation

For an MoE model like Qwen3.5-122B-A10B: each token only activates ~10B of the 122B total parameters. At NVFP4 (~0.5 bytes/param), that's **~5GB read per token**. With 546 GB/s bandwidth:

```
546 GB/s ÷ 5 GB/token ≈ 110 tok/s theoretical ceiling
```

In practice, routing overhead and memory access patterns bring this down to approximately 40-50 tok/s on M4 Max 128GB — comfortably fast for interactive work.

### 2.4 KV Cache: The Memory That Grows With Context

Notice the "cache K, V" step in the diagram above. Every time you generate a token, the model needs to "remember" all previous tokens in the context. It does this by caching the **Key** and **Value** matrices for each layer — the KV cache. Without it, generating token N would require recomputing attention over all N-1 previous tokens from scratch.

**The memory formula** (burn this into your brain):

```
KV cache memory = 2 × num_layers × num_kv_heads × head_dim × seq_len × bytes_per_element
```

The "2" is for Keys and Values (one tensor each, same size). `num_layers` is how many transformer blocks are stacked. `num_kv_heads` and `head_dim` come from the attention mechanism. `seq_len` is your context length — the dangerous variable, because it grows with every token. `bytes_per_element` is the storage precision (bf16 = 2 bytes by default in MLX).

**Worked example — Qwen3.5-35B-A3B at 32K context (bf16 KV cache):**

> **Note:** Qwen3.5-35B-A3B is a hybrid model where only 10 of its 40 layers use GQA (full attention); the other 30 use Gated DeltaNet (linear attention with no standard KV cache). The simplified example below uses `num_layers = 40` to show the formula mechanics — see Module 1B for the correct hybrid formula.

```
num_layers    = 40   (simplified: all 40 as GQA for formula demo)
num_kv_heads  = 2    (GQA — only 2 KV heads shared across all query heads)
head_dim      = 256
seq_len       = 32,768  (32K tokens)
bytes_per_elem = 2   (bf16)

KV cache = 2 × 40 × 2 × 256 × 32,768 × 2
         = 2,684,354,560 bytes
         ≈ 2.68 GB  (simplified 40-layer formula)

Correct (10 GQA layers only):
KV cache = 2 × 10 × 2 × 256 × 32,768 × 2
         = 671,088,640 bytes
         ≈ 0.67 GB
```

This is why long multi-turn sessions eat memory even though the model weights never change.

**Grouped Query Attention (GQA):** Modern models use GQA, where multiple query heads share a single key/value head pair. Always check the model's `config.json` for `num_key_value_heads` (not `num_attention_heads`) when estimating KV cache memory.

**Key Insight:** KV cache is why long conversations eat memory. The model weights are static — they load once. But every new token in context costs you. Budget for both: `model_memory + kv_cache_memory < total_RAM`.

**Deterministic replay:** If you load a model fresh and send it the exact same prompt, the KV cache state will be *identical* every time — same keys, same values, byte for byte. The model is a pure function of its weights + input. There's no hidden state carrying over between sessions. With temperature=0 (greedy decoding), you get the same output tokens too. This means you can write deterministic regression tests, reproduce bugs exactly, and reason about the model as a mathematical function rather than a mysterious oracle.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8800/v1", api_key="unused")

response = client.chat.completions.create(
    model="mlx-community/Qwen3-32B-4bit",
    messages=[{"role": "user", "content": "Explain KV caching in one paragraph."}],
    max_tokens=200,
)
print(response.choices[0].message.content)

### 2.5 Prefill vs Decode: Two Very Different Operations

Inference has two phases, and they behave completely differently:

```
PREFILL PHASE                    DECODE PHASE
─────────────────────────────    ────────────────────────────
Process the entire input prompt  Generate one token at a time
Highly parallel (like training)  Sequential — each token
GPU-bound, fast                  depends on previous
Measured in: TTFT                Memory-bandwidth-bound, slow
(Time To First Token)            Measured in: TPS
                                 (Tokens Per Second)
```

**The pipeline analogy:** Prefill is like flashing firmware — you write a whole block at once, the hardware handles it in burst mode. Decode is like bit-banging a GPIO — one bit at a time, you're waiting on every cycle. They require completely different optimization strategies.

This is why a model can feel fast at first (prefill completes quickly) but slow at sustained generation (decode is memory-bandwidth-limited by how fast you can read those billions of weights from RAM).

> **Try this:** Send a long prompt and watch: the first token appears after a noticeable delay (prefill processing your entire input), then subsequent tokens stream at a steady cadence (decode). The pause-then-stream pattern is prefill vs decode in action.

### 2.6 Mixture of Experts: More Parameters, Less Compute

MoE models (like Qwen3.5-122B-A10B or Mixtral) have a trick: instead of one large feed-forward network (FFN) per layer, they have N small "expert" FFNs and a router that picks a subset of them per token.

```
Dense model FFN:           MoE FFN:
                           ┌─Expert 1─┐
                           ├─Expert 2─┤
x ──→ [Big FFN] ──→ y     x ──→ [Router] ──→ ├─Expert 3─┤ ──→ y
                           ├─Expert 4─┤
                           └─Expert 5─┘ (only 2 activated)
       All params used         Only 2/N params used per token
```

**How the router works:** The router is a small learned linear layer that produces a score for each expert. The top-K experts (usually 2) by score are selected, their outputs are computed, and the final result is a weighted sum of those K outputs.

**Load imbalance:** A known failure mode is expert collapse — many tokens routing to the same 1–2 experts, leaving others idle. MoE training uses auxiliary load-balancing losses to encourage even distribution across experts.

**When MoE underperforms:** Dense models often outperform MoE on deep reasoning chains (math competition problems, multi-step proofs). The routing mechanism can fragment reasoning across expert boundaries in ways that a dense model's continuous computation path does not.

**Key Insight:** An MoE model can have 122B total parameters but only "use" 10B per token (hence "A10B" — 122B total, 10B active). All 122B weights must fit in RAM (~65GB at NVFP4), but inference speed is determined by the active 10B (~5GB), not all 122B. You get big-model quality at smaller-model inference cost.

### 2.7 Model Evaluation: Choosing What to Download

There are thousands of models on HuggingFace. Here's the 6-step workflow for deciding which ones are worth downloading:

1. **Find candidates.** Query HuggingFace for trending MLX models. Filter by format (safetensors, MLX-native) and size (will it fit your RAM?). The `mlx-community` organization maintains pre-quantized MLX versions of most popular models.

2. **Check RAM fit.** Apply the **60% rule** — keep model weights under 60% of total RAM (~77GB on 128GB) for stable long conversations. KV cache needs the remaining headroom.

3. **Cross-reference benchmarks.** Check the Open LLM Leaderboard for standardized rankings (ARC, HellaSwag, MMLU, GSM8K, IFEval). Look for models that score well across the mix, not just the headline number. Don't trust benchmarks alone — some models are trained specifically to score well without generalizing.

4. **Check community feedback.** Search r/LocalLLaMA and r/LocalLLM on Reddit for real user reports. These catch things benchmarks miss.

5. **Check human preference scores.** LMArena (formerly Chatbot Arena) provides ELO ratings based on blind human comparisons. The divergence between benchmark rank and arena ELO is itself a signal.

6. **Decide.** Does this model fill a gap in your lineup? Download with rationale, not just because it's new.

### Worked Comparison

| Signal | Qwen3.5-122B-A10B | DenseX-70B |
|---|---|---|
| RAM fit | 65 GB — tight but fits | 35 GB — comfortable |
| MMLU (knowledge) | 90.1 | 88.5 |
| IFEval (instruction) | 84.2 | 77.1 |
| r/LocalLLaMA | "slow but impressive quality" | "good at facts, stiff" |
| LMArena ELO | 1380 | 1240 |

The 122B MoE model wins on both quality and speed (MoE activates only 10B params per token, so it's faster than the 70B dense model that reads all 35GB per token). The main tradeoff is RAM: 65GB leaves less headroom for KV cache on long contexts.

## 3. Hands-On: First Inference

You now have the concepts — unified memory, quantization, layers, KV cache, prefill/decode, MoE. Time to run things.

### Step 1: Verify MLX Installation

In [ ]:
import mlx.core as mx

print(f"MLX device: {mx.default_device()}")
print(f"MLX version: {mx.__version__}")

### Step 2: Your First Inference Call

Your MLX server should be running on port 8800. Every code example talks to that server via the OpenAI-compatible API — no model reloading, instant responses.

`mlx_lm.server` makes your local model a **drop-in replacement for the OpenAI API**. Every library and tool that speaks OpenAI protocol (LangChain, LlamaIndex, Cursor, etc.) can point to `localhost:8800` and use your local model instead.

In [ ]:
from openai import OpenAI
import time

client = OpenAI(base_url="http://localhost:8800/v1", api_key="unused")

prompt = "Explain transformer attention in one paragraph, using an analogy."
t0 = time.time()

response = client.chat.completions.create(
    model="mlx-community/Qwen3-32B-4bit",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
)

elapsed = time.time() - t0
text = response.choices[0].message.content
tokens = response.usage.completion_tokens
print(f"Generated {tokens} tokens in {elapsed:.1f}s ({tokens/elapsed:.1f} tok/s)")
print(f"\nResponse:\n{text}")

## 4. Hands-On: Benchmark

MLX uses lazy evaluation and JIT-style compilation. The first run includes compilation overhead. **Always do one warmup pass before timing.**

**Common Pitfall:** Not warming up before benchmarking. The first request includes connection setup and JIT overhead — your numbers will be wrong if you time it.

In [ ]:
from openai import OpenAI
import time

client = OpenAI(base_url="http://localhost:8800/v1", api_key="unused")

# Warm up
client.chat.completions.create(
    model="mlx-community/Qwen3-32B-4bit",
    messages=[{"role": "user", "content": "hi"}],
    max_tokens=5
)

# Benchmark
prompt = "Write a haiku about silicon."
t0 = time.time()
response = client.chat.completions.create(
    model="mlx-community/Qwen3-32B-4bit",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=100,
)
elapsed = time.time() - t0
tokens = response.usage.completion_tokens
tps = tokens / elapsed
print(f"{tps:.1f} tok/s ({tokens} tokens in {elapsed:.1f}s)")
print(f"Response: {response.choices[0].message.content}")

## 5. Hands-On: Sampling Parameters

**Temperature** is the most important parameter you'll tune:
- `temp=0.0` → **Greedy decoding**, always picks the highest-probability token. Deterministic. Good for structured output, code, math.
- `temp=0.7` → **Balanced** creativity. Good default for chat.
- `temp=1.0-1.2` → **High variance** output. More creative, sometimes surprising. Above ~1.3 on large MoE models tends to produce gibberish.

**Top-p (nucleus sampling):** Only sample from the smallest set of tokens whose cumulative probability >= p. At `top_p=0.9`, you throw away the long tail of unlikely tokens. Reduces nonsense without hurting creativity much.

**Top-k:** Only sample from the k most likely tokens. Cruder than top-p but fast.

Let's see the difference in practice:

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8800/v1", api_key="unused")

prompt = "Continue this story: The robot opened its eyes for the first time and saw"

configs = [
    {"temp": 0.0, "label": "Greedy (deterministic)"},
    {"temp": 0.7, "label": "Balanced (default-ish)"},
    {"temp": 1.2, "label": "Creative (chaotic)"},
]

for cfg in configs:
    print(f"\n--- {cfg['label']} (temp={cfg['temp']}) ---")
    response = client.chat.completions.create(
        model="mlx-community/Qwen3-32B-4bit",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=cfg["temp"],
    )
    print(response.choices[0].message.content)

## 6. Memory Monitoring

**Common Pitfall:** Running out of memory silently. macOS will start swapping to NVMe before crashing. You'll see tok/s drop to 1-3 as the system pages.

**Threshold heuristic:** When `mx.metal.get_active_memory()` exceeds ~90% of your physical RAM (e.g., >115 GB on a 128 GB machine), you are in swap territory and throughput will degrade sharply.

**Remediation options** (in order of least to most disruptive):
1. **Cap the KV cache:** restart with `--max-kv-size 4096`
2. **Reduce context:** shorten your prompt or summarize conversation history
3. **Drop to a lower quantization:** switch from 8-bit to 4-bit
4. **Load a smaller model**

In [ ]:
import mlx.core as mx

active = mx.metal.get_active_memory() / 1e9
peak = mx.metal.get_peak_memory() / 1e9
print(f"Active GPU memory: {active:.2f} GB")
print(f"Peak GPU memory:   {peak:.2f} GB")

import psutil
ram = psutil.virtual_memory()
print(f"\nSystem RAM: {ram.total / 1e9:.1f} GB total, {ram.available / 1e9:.1f} GB available")
print(f"RAM usage:  {ram.percent}%")

## 7. Key Takeaways

The "aha" moments from this module:

1. **Unified memory is the superpower.** The reason you can run a 122B MoE model locally at NVFP4 isn't magic — it's that there's no VRAM bottleneck. Your 128GB is accessible to GPU, CPU, and Neural Engine simultaneously, from the same address space.

2. **Inference is memory-bandwidth-bound, not compute-bound.** Every token you generate requires reading the active model weights from RAM. Your throughput ceiling is `memory_bandwidth / active_weights_per_token`. Quantization is powerful because it shrinks model size, directly improving throughput.

3. **A transformer is layers of attention + FFN, stacked deep.** Layers are the repeating blocks. Heads are parallel attention detectors inside each layer. Every weight matrix in every layer must be read from RAM for every token.

4. **KV cache memory grows with context length.** Static model weights + dynamic KV cache = your two memory consumers. Budget for both. Use `--max-kv-size` to cap the latter.

5. **Prefill and decode are fundamentally different.** Prefill is parallel and fast (like flashing firmware). Decode is sequential and slow (like bit-banging a GPIO). TTFT and TPS measure these two phases separately — don't conflate them.

6. **NVFP4 is the quantization format for large models.** NVFP4's floating-point format (E2M1 weights, E4M3 scales, group_size=16) delivers better quality than affine INT4 at the same bit width by placing more representable values near zero where weights cluster.

7. **`mlx_lm.server` makes everything OpenAI-compatible.** Any library or tool that calls the OpenAI API can point to `localhost:8800` and use your local model — with zero code changes.

8. **The model is a pure function.** Same weights + same input = same KV cache state, every time. No hidden memory between sessions. This makes LLMs debuggable and testable in ways that feel familiar from embedded systems.

## 8. What's Next

You can now load any open-source LLM, run it locally, measure its performance, and serve it via a standard API. That's the inference foundation everything else builds on.

- **Module 1B: Qwen3.5 Model Datasheet** — dives into the specific model architecture: hybrid DeltaNet/GQA architecture, revised KV cache formula for hybrid models, and MoE variants. Think of it as the datasheet review before you start writing firmware.

- **Module 1C: Inference Optimization** — speculative decoding, prompt prefix caching, advanced quantization formats (NVFP4/MXFP4), sliding window attention, and continuous batching.

- **Module 2: Tokenization & Embeddings** — how does the model actually see text? Tokenizers chop text into subword pieces, each mapped to an integer ID. Embeddings map those IDs into high-dimensional vector spaces where "meaning" becomes geometric distance.

---

🔔 **Subscribe** to get notified when the next module drops.

*Module 1 complete. Go run something large and local.*

## Sources & References

1. [Apple MacBook Pro Tech Specs](https://www.apple.com/macbook-pro/specs/) — M4 Max memory bandwidth specifications (546 GB/s for 40-core GPU variant)
2. [TechPowerUp H100 SXM5 Specs](https://www.techpowerup.com/gpu-specs/h100-sxm5-80-gb.c3900) — H100 SXM5 memory bandwidth (3.35 TB/s, HBM3)
3. [Hyperstack: H100 PCIe vs SXM Performance](https://www.hyperstack.cloud/technical-resources/performance-benchmarks/comparing-nvidia-h100-pcie-vs-sxm-performance-use-cases-and-more) — PCIe (~2 TB/s) vs SXM5 H100 comparison
4. [Apple M4 – Wikipedia](https://en.wikipedia.org/wiki/Apple_M4) — M4 chip family specs: Neural Engine core count (16-core), TOPS rating (38 TOPS)
5. [MLX GitHub Repository](https://github.com/ml-explore/mlx) — Framework documentation, lazy evaluation design, and unified memory architecture
6. [WWDC25 – Get started with MLX for Apple Silicon](https://developer.apple.com/videos/play/wwdc2025/315/) — Apple's official MLX deep-dive: lazy evaluation, dynamic graphs, GPU scheduling
7. [GPTQ: Accurate Post-Training Quantization (arXiv 2210.17323)](https://arxiv.org/abs/2210.17323) — Group quantization methodology
8. [SqueezeLLM: Dense-and-Sparse Quantization (arXiv 2306.07629)](https://arxiv.org/abs/2306.07629) — Perplexity degradation curves across bit widths
9. [AQLM: Extreme Compression via Additive Quantization (arXiv 2401.06118)](https://arxiv.org/abs/2401.06118) — Quality cliff at low bit widths
10. [QuIP#: LLM Quantization with Hadamard Incoherence (arXiv 2402.04396)](https://arxiv.org/abs/2402.04396) — Near-lossless quantization above ~4.5 bits
11. [GLU Variants Improve Transformer (arXiv 2002.05202)](https://arxiv.org/abs/2002.05202) — SwiGLU activation function
12. [GQA: Training Generalized Multi-Query Transformer Models (arXiv 2305.13245)](https://arxiv.org/abs/2305.13245) — Grouped Query Attention mechanism
13. [Mixtral of Experts (arXiv 2401.04088)](https://arxiv.org/abs/2401.04088) — Mixtral MoE architecture
14. [Qwen3 Blog](https://qwenlm.github.io/blog/qwen3/) — Qwen3 MoE architecture, load-balancing, benchmarks
15. [Qwen3.5-122B-A10B – Hugging Face](https://huggingface.co/Qwen/Qwen3.5-122B-A10B) — Model card: 122B total parameters, ~10B active per token
16. [Qwen3.5 Blog](https://qwenlm.github.io/blog/qwen3-5/) — Hybrid DeltaNet/GQA architecture details
17. [LMArena](https://lmarena.ai) — Human preference ELO leaderboard (blind pairwise comparisons)
18. [Artificial Analysis](https://artificialanalysis.ai) — Independent benchmark and throughput data for LLM models
19. [MLX Sharp Bits / Gotchas](https://ml-explore.github.io/mlx/dev/usage/sharp_bits.html) — Lazy evaluation pitfalls and memory management
20. [mlx-lm GitHub (mlx-examples/llms)](https://github.com/ml-explore/mlx-examples/tree/main/llms) — mlx_lm source code, CLI tools, usage examples
21. [mlx_lm README](https://github.com/ml-explore/mlx-examples/blob/main/llms/README.md) — Canonical CLI reference for convert, server, and generation flags

- See **Module 1B** for Qwen3.5-specific architecture sources (model card, DeltaNet paper, Triton kernels)
- See **Module 1C** for quantization format sources (NVFP4, MXFP4, E2M1 specifications)